In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
from utils import check_group_holidays

In [ ]:
sample_size = 25

In [ ]:
current_dir = os.getcwd()

data_dir = "tech_challenge-data-us_flights_ml"

In [ ]:
flights_csv = "flights.csv"
df_flights = pd.read_csv(os.path.join(current_dir, data_dir, flights_csv))

In [ ]:
df_flights.info()

In [ ]:
not_cancelled_mask = df_flights["CANCELLED"] == 0

df_flights.loc[not_cancelled_mask, "FLIGHT_SEQUENCE"] = (
    df_flights.loc[not_cancelled_mask, ["TAIL_NUMBER", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"]]
    .sort_values(by=["TAIL_NUMBER", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"])
    .groupby(["TAIL_NUMBER", "YEAR", "MONTH", "DAY"])
    .cumcount() + 1
)

df_flights["FLIGHT_SEQUENCE"] = df_flights["FLIGHT_SEQUENCE"].astype("Int64")

In [ ]:
df_flights_considered = df_flights.loc[
    (df_flights["CANCELLED"] == 0) & (df_flights["DIVERTED"] == 0) & (df_flights["MONTH"] != 10),
    [
        "YEAR", "MONTH", "DAY", "DAY_OF_WEEK",
        "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "FLIGHT_SEQUENCE",
        "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DISTANCE",
        "SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL", "SCHEDULED_TIME",
        "ARRIVAL_DELAY"
    ]
]

In [ ]:
del df_flights
gc.collect()

In [ ]:
df_flights_considered["HOUR"] = df_flights_considered["SCHEDULED_DEPARTURE"] // 100

wd_mapping = {
    1: "MON",
    2: "TUE",
    3: "WED",
    4: "THU",
    5: "FRI",
    6: "SAT",
    7: "SUN",
}
df_flights_considered["DAY_OF_WEEK"] = pd.Categorical(
    df_flights_considered["DAY_OF_WEEK"],
    categories=wd_mapping.keys(),
    ordered=True
)
df_flights_considered["DAY_OF_WEEK_ABBR"] = df_flights_considered["DAY_OF_WEEK"].cat.rename_categories(wd_mapping)

df_flights_considered["ROUTE"] = df_flights_considered["ORIGIN_AIRPORT"].astype(str) + "-" + df_flights_considered["DESTINATION_AIRPORT"].astype(str)

tol_min = 15
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] >= tol_min

In [ ]:
df_flights_considered.info()

In [ ]:
df_flights_considered.sample(n=sample_size)

## Sampled dataset

In [ ]:
rng = np.random.default_rng(seed=42)

n_rows, _ = df_flights_considered.shape
sampled_dataset_mask = np.zeros(n_rows, dtype=bool)

sampled_dataset_size = 100_000
sampled_dataset_indices = rng.choice(n_rows, size=sampled_dataset_size, replace=False)

sampled_dataset_mask[sampled_dataset_indices] = True

In [ ]:
df_flights_sampled = df_flights_considered[sampled_dataset_mask].copy(deep=True)

In [ ]:
df_flights_sampled.info()

In [ ]:
df_flights_sampled.sample(n=sample_size)

In [ ]:
df_flights_sampled["FLIGHT_SEQUENCE"] = df_flights_sampled["FLIGHT_SEQUENCE"].astype(int)
df_flights_sampled["ORIGIN_AIRPORT"] = df_flights_sampled["ORIGIN_AIRPORT"].astype(str)
df_flights_sampled["DESTINATION_AIRPORT"] = df_flights_sampled["DESTINATION_AIRPORT"].astype(str)
df_flights_sampled["SCHEDULED_TIME"] = df_flights_sampled["SCHEDULED_TIME"].astype(int)
df_flights_sampled["ARRIVAL_DELAY"] = df_flights_sampled["ARRIVAL_DELAY"].astype(int)

In [ ]:
plot_opts = dict(
    kind="bar",
    stacked=True,
    color=["deepskyblue", "gainsboro"],
)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["MONTH", "DELAYED"]

df_flights_considered[cols].value_counts().unstack().reindex(range(1, 13), fill_value=0).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack().reindex(range(1, 13), fill_value=0).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DAY", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DAY_OF_WEEK_ABBR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["HOUR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["ORIGIN_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["DESTINATION_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["ROUTE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["AIRLINE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["FLIGHT_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["TAIL_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

cols = ["FLIGHT_SEQUENCE", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axc)
df_flights_sampled[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts, ax=axs)

In [ ]:
del df_flights_considered
gc.collect()

## Enriquecimento e variáveis derivadas

In [ ]:
airports_csv = "airports.csv"
df_airports = pd.read_csv(os.path.join(current_dir, data_dir, airports_csv))

In [ ]:
df_airports.info()

In [ ]:
df_airports.sample(n=sample_size)

In [ ]:
airports_cols = ["IATA_CODE", "CITY", "STATE", "COUNTRY", "LATITUDE", "LONGITUDE"]

origin_airport_cols = {c: "ORIGIN_" + c for c in airports_cols[1:]}
destination_airport_cols = {c: "DESTINATION_" + c for c in airports_cols[1:]}

df_flights_sampled = df_flights_sampled.merge(
    right=df_airports[airports_cols].rename(columns={"IATA_CODE": "ORIGIN_AIRPORT", **origin_airport_cols}),
    how="left",
    on="ORIGIN_AIRPORT",
    validate="many_to_one"
).merge(
    right=df_airports[airports_cols].rename(columns={"IATA_CODE": "DESTINATION_AIRPORT", **destination_airport_cols}),
    how="left",
    on="DESTINATION_AIRPORT",
    validate="many_to_one"
)

In [ ]:
df_flights_sampled["DT_SCHEDULED_DEPARTURE"] = pd.to_datetime(df_flights_sampled[["YEAR", "MONTH", "DAY"]], format="%Y-%m-%d")

df_flights_sampled = (
    df_flights_sampled
    .groupby(["ORIGIN_COUNTRY", "ORIGIN_STATE"], group_keys=True)
    .apply(check_group_holidays)
    .reset_index(level=[0, 1], drop=False)
    .rename(columns={"HOLIDAY": "HOLIDAY_ORIGIN", "HOLIDAY_EVE": "HOLIDAY_EVE_ORIGIN"})
)

df_flights_sampled = (
    df_flights_sampled
    .groupby(["DESTINATION_COUNTRY", "DESTINATION_STATE"], group_keys=True)
    .apply(check_group_holidays)
    .reset_index(level=[0, 1], drop=False)
    .rename(columns={"HOLIDAY": "HOLIDAY_DESTINATION", "HOLIDAY_EVE": "HOLIDAY_EVE_DESTINATION"})
)

In [ ]:
df_flights_sampled["DEPARTURE_ACC_MINUTES"] = (df_flights_sampled["SCHEDULED_DEPARTURE"] // 100) * 60 + (df_flights_sampled["SCHEDULED_DEPARTURE"] % 100)

df_flights_sampled["ARRIVAL_ACC_MINUTES"] = (df_flights_sampled["SCHEDULED_ARRIVAL"] // 100) * 60 + (df_flights_sampled["SCHEDULED_ARRIVAL"] % 100)

In [ ]:
km_to_miles_const = 0.621371

dist_cat_bins = [
    0,
    1000 * km_to_miles_const,
    3000 * km_to_miles_const,
    np.inf,
]

dist_cat_labels = [
    "Short Distance",
    "Medium Distance",
    "Long Distance",
]

df_flights_sampled["DISTANCE_CATEGORY"] = pd.cut(
    df_flights_sampled["DISTANCE"],
    bins=dist_cat_bins,
    right=False,
    labels=dist_cat_labels,
    ordered=True
)

In [ ]:
sch_time_cat_bins = [
    0,
    1 * 60,
    3 * 60,
    5 * 60,
    np.inf,
]

sch_time_cat_labels = [
    "Short Duration",
    "Medium Duration",
    "Long Duration",
    "Very Long Duration",
]

df_flights_sampled["SCHEDULED_TIME_CATEGORY"] = pd.cut(
    df_flights_sampled["SCHEDULED_TIME"],
    bins=sch_time_cat_bins,
    right=False,
    labels=sch_time_cat_labels,
    ordered=True
)

## Limpeza

In [ ]:
df_flights_sampled["ORIGIN_COUNTRY"].unique()

In [ ]:
df_flights_sampled["DESTINATION_COUNTRY"].unique()

In [ ]:
df_flights_sampled.drop(
    columns=[
        "YEAR", "DAY_OF_WEEK_ABBR", "SCHEDULED_DEPARTURE", "DT_SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL",
        "FLIGHT_NUMBER", "TAIL_NUMBER",
        "ORIGIN_COUNTRY", "DESTINATION_COUNTRY",
        "DELAYED",
    ],
    inplace=True
)

In [ ]:
df_flights_sampled.info()

In [ ]:
df_flights_sampled.sample(n=sample_size)

In [ ]:
u_cat_cols = [
    "AIRLINE",
    "ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "DESTINATION_CITY", "DESTINATION_STATE",
    "ROUTE",
]

df_flights_sampled[u_cat_cols] = df_flights_sampled[u_cat_cols].astype("category")

In [ ]:
df_flights_sampled["MONTH"] = pd.Categorical(
    df_flights_sampled["MONTH"],
    categories=list(range(1, 13)),
    ordered=True
)

df_flights_sampled["DAY"] = pd.Categorical(
    df_flights_sampled["DAY"],
    categories=list(range(1, 32)),
    ordered=True
)

df_flights_sampled["HOUR"] = pd.Categorical(
    df_flights_sampled["HOUR"],
    categories=list(range(0, 24)),
    ordered=True
)

In [ ]:
df_flights_sampled.info()

## Carga

In [ ]:
flights_sampled_parquet = "flights_sampled.parquet"
df_flights_sampled.to_parquet(os.path.join(current_dir, flights_sampled_parquet), engine="fastparquet", index=False)